# Income Classification — Extended Project

Building on the base logistic regression pipeline, this notebook extends the project with:

1. **Deeper EDA** — univariate & bivariate distributions, missing-value audit
2. **Feature engineering** — interaction terms, binning age/hours, log-transform capital-gain
3. **Scaling comparison** — raw vs StandardScaler vs MinMaxScaler
4. **Regularisation sweep** — grid-search over C values, L1 vs L2 penalty
5. **Cross-validation** — stratified k-fold ROC-AUC
6. **Class-imbalance strategies** — class_weight, SMOTE, undersampling
7. **Threshold tuning** — optimise for F1, precision-recall trade-off
8. **Model comparison** — Logistic Regression vs Random Forest vs Gradient Boosting
9. **SHAP / coefficient interpretation** with confidence intervals via bootstrap
10. **Deployment-ready pipeline** using `Pipeline` + `ColumnTransformer`

In [ ]:
# ── Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, precision_recall_curve
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

---
## 1. Data Loading & Initial Inspection

In [ ]:
col_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]
df = pd.read_csv('adult.data', header=None, names=col_names)

# Strip whitespace from object columns
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].str.strip()

print(df.shape)
df.head()

In [ ]:
# Missing-value audit (UCI adult uses '?' for missing)
df.replace('?', np.nan, inplace=True)
missing = df.isna().sum()
print(missing[missing > 0])

# Quick summary
df.describe(include='all')

---
## 2. Deeper EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Age distribution by income
sns.histplot(data=df, x='age', hue='income', kde=True, ax=axes[0, 0])

# Hours per week by income
sns.boxplot(data=df, x='income', y='hours-per-week', ax=axes[0, 1])

# Education count
df['education'].value_counts().plot(kind='barh', ax=axes[0, 2])

# Capital gain log-scale
df['log_capital_gain'] = np.log1p(df['capital-gain'])
sns.histplot(data=df, x='log_capital_gain', hue='income', ax=axes[1, 0])

# Workclass vs income proportion
pd.crosstab(df['workclass'], df['income'], normalize='index').plot(
    kind='barh', stacked=True, ax=axes[1, 1]
)

# Sex vs income
pd.crosstab(df['sex'], df['income'], normalize='index').plot(
    kind='bar', stacked=True, ax=axes[1, 2]
)

plt.tight_layout()
plt.show()

In [ ]:
# Class imbalance
print(df['income'].value_counts(normalize=True))

---
## 3. Feature Engineering

In [ ]:
# Age bins
df['age_bin'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 65, 100],
                       labels=['17-25', '26-35', '36-45', '46-55', '56-65', '65+'])

# Hours bins
df['hours_bin'] = pd.cut(df['hours-per-week'], bins=[0, 20, 35, 40, 50, 99],
                         labels=['part-time', 'under-35', 'full-time', 'over-40', 'over-50'])

# Log-transform skewed capital features
df['log_capital_gain'] = np.log1p(df['capital-gain'])
df['log_capital_loss'] = np.log1p(df['capital-loss'])

# Interaction: education_num × age
df['edu_x_age'] = df['education-num'] * df['age']

# Has-capital-gain indicator
df['has_capital_gain'] = (df['capital-gain'] > 0).astype(int)

df[['log_capital_gain', 'log_capital_loss', 'edu_x_age', 'has_capital_gain']].head()

---
## 4. Prepare X and y with New Features

In [ ]:
feature_cols = [
    'age', 'log_capital_gain', 'log_capital_loss', 'hours-per-week',
    'sex', 'race', 'education', 'age_bin', 'hours_bin',
    'has_capital_gain', 'edu_x_age'
]

X = pd.get_dummies(df[feature_cols], drop_first=True)
y = np.where(df['income'] == '<=50K', 0, 1)

print(f'X shape: {X.shape}')
print(f'y class balance: {np.bincount(y) / len(y)}')

---
## 5. Scaling Comparison

In [ ]:
scalers = {
    'raw':       None,
    'standard':  StandardScaler(),
    'minmax':    MinMaxScaler(),
}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

for name, scaler in scalers.items():
    model = LogisticRegression(C=0.05, penalty='l1', solver='liblinear', max_iter=1000)
    pipe = Pipeline([('scaler', scaler), ('model', model)]) if scaler else Pipeline([('model', model)])
    pipe.fit(X_train, y_train)
    score = pipe.score(X_test, y_test)
    print(f'{name:12s} → Accuracy: {score:.4f}')

---
## 6. Regularisation Sweep (GridSearchCV)

In [ ]:
param_grid = {
    'model__C':       [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
    'model__penalty': ['l1', 'l2'],
}

base_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(solver='liblinear', max_iter=2000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(base_pipe, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print(f'Best params: {grid.best_params_}')
print(f'Best CV AUC: {grid.best_score_:.4f}')

---
## 7. Cross-Validation AUC

In [ ]:
best_model = grid.best_estimator_
cv_auc = cross_val_score(best_model, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'5-Fold CV AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

---
## 8. Class-Imbalance Strategies

In [ ]:
# --- Strategy A: class_weight='balanced' ---
model_balanced = LogisticRegression(
    C=grid.best_params_['model__C'],
    penalty=grid.best_params_['model__penalty'],
    solver='liblinear', class_weight='balanced', max_iter=2000
)
pipe_bal = Pipeline([('scaler', StandardScaler()), ('model', model_balanced)])
pipe_bal.fit(X_train, y_train)
y_pred_bal = pipe_bal.predict(X_test)
print('Balanced class_weight F1:', f1_score(y_test, y_pred_bal))
print(classification_report(y_test, y_pred_bal))

# --- Strategy B: SMOTE (requires imblearn) ---
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=RANDOM_STATE)
# X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
# print('After SMOTE:', np.bincount(y_resampled))
# model_smote = LogisticRegression(C=0.05, penalty='l1', solver='liblinear')
# model_smote.fit(X_resampled, y_resampled)
# print('SMOTE F1:', f1_score(y_test, model_smote.predict(X_test)))

# --- Strategy C: Random Undersampling ---
# from imblearn.under_sampling import RandomUnderSampler
# rus = RandomUnderSampler(random_state=RANDOM_STATE)
# X_rus, y_rus = rus.fit_resample(X_train, y_train)
# model_rus = LogisticRegression(C=0.05, penalty='l1', solver='liblinear')
# model_rus.fit(X_rus, y_rus)
# print('Undersampling F1:', f1_score(y_test, model_rus.predict(X_test)))

---
## 9. Threshold Tuning

In [ ]:
y_prob = best_model.predict_proba(X_test)[:, 1]

# Precision-Recall vs Threshold
prec, rec, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * prec * rec / (prec + rec + 1e-12)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f'Optimal threshold for F1: {best_threshold:.4f}')
print(f'F1 at optimal threshold: {f1_scores[best_idx]:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(thresholds, prec[:-1], label='Precision')
ax[0].plot(thresholds, rec[:-1], label='Recall')
ax[0].plot(thresholds, f1_scores[:-1], label='F1')
ax[0].axvline(best_threshold, color='red', linestyle='--', label=f'Threshold={best_threshold:.3f}')
ax[0].set_xlabel('Threshold')
ax[0].legend()
ax[0].set_title('Precision / Recall / F1 vs Threshold')

fpr, tpr, roc_thresh = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)
ax[1].plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.4f})')
ax[1].plot([0, 1], [0, 1], '--', color='grey')
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('ROC Curve')
ax[1].legend()

plt.tight_layout()
plt.show()

---
## 10. Model Comparison — LR vs RF vs GBM

In [ ]:
models = {
    'LogReg':  Pipeline([('scaler', StandardScaler()),
                         ('model', LogisticRegression(C=0.05, penalty='l1',
                           solver='liblinear', max_iter=2000))]),
    'RF':      Pipeline([('model', RandomForestClassifier(
                           n_estimators=200, max_depth=12,
                           random_state=RANDOM_STATE, n_jobs=-1))]),
    'GBM':     Pipeline([('model', GradientBoostingClassifier(
                           n_estimators=200, max_depth=3,
                           learning_rate=0.1, random_state=RANDOM_STATE))]),
}

results = []
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    preds = mdl.predict(X_test)
    proba = mdl.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, proba),
    })

results_df = pd.DataFrame(results).set_index('Model').round(4)
print(results_df)

---
## 11. Coefficient Interpretation with Bootstrap Confidence Intervals

In [ ]:
lr_final = models['LogReg'].named_steps['model']
scaler_final = models['LogReg'].named_steps['scaler']

coef_df = pd.DataFrame({
    'var': X_train.columns,
    'coef': lr_final.coef_[0]
}).sort_values('coef')
coef_df = coef_df[coef_df['coef'].abs() > 0]

# Bootstrap CI
n_boot = 200
boot_coefs = []
for _ in range(n_boot):
    idx = np.random.choice(len(X_train), len(X_train), replace=True)
    Xb = scaler_final.transform(X_train.iloc[idx])
    yb = y_train[idx]
    m = LogisticRegression(C=0.05, penalty='l1', solver='liblinear', max_iter=2000)
    m.fit(Xb, yb)
    boot_coefs.append(m.coef_[0])

boot_arr = np.array(boot_coefs)
coef_df['ci_low'] = np.percentile(boot_arr, 2.5, axis=0)[coef_df.index.values]
coef_df['ci_high'] = np.percentile(boot_arr, 97.5, axis=0)[coef_df.index.values]

plt.figure(figsize=(12, 7))
plt.barh(coef_df['var'], coef_df['coef'],
         xerr=[coef_df['coef'] - coef_df['ci_low'],
               coef_df['ci_high'] - coef_df['coef']],
         capsize=3)
plt.title('Logistic Regression Coefficients with 95% Bootstrap CI')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

---
## 12. Deployment-Ready Pipeline with ColumnTransformer

In [ ]:
numeric_features = ['age', 'log_capital_gain', 'log_capital_loss',
                   'hours-per-week', 'edu_x_age', 'has_capital_gain']
categorical_features = ['sex', 'race', 'education', 'age_bin', 'hours_bin']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
])

deploy_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(C=0.05, penalty='l1',
                                       solver='liblinear', max_iter=2000))
])

deploy_pipe.fit(df[numeric_features + categorical_features], y)
print('Pipeline ready for deployment.')
print(deploy_pipe)

---
## 13. Summary

| Extension | Key Finding |
|---|---|
| Scaling | Minimal accuracy impact for LR with L1; recommended for interpretability stability |
| Regularisation sweep | Optimal C varies; L1 performs feature selection naturally |
| Class imbalance | `class_weight='balanced'` improves recall for >50K class at cost of precision |
| Threshold tuning | Default 0.5 is rarely optimal; tune based on business cost matrix |
| Model comparison | GBM typically outperforms LR on AUC but sacrifices interpretability |
| Bootstrap CI | Some coefficients have wide CIs — interpret with caution |